In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import make_pipeline
import sys
import os
sys.path.append("/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis")
import preprocessing

In [4]:
%pwd

'/data/resources/weichel-llama3/work'

In [6]:
X_train = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/synthetic_data/data_20251219__level_1__subclasses_None/train_data.csv")["text"].to_list()
y_train = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/synthetic_data/data_20251219__level_1__subclasses_None/train_data.csv")["label"].to_list()
X_test = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/synthetic_data/data_20251219__level_1__subclasses_None/test_data.csv")["text"].to_list()
y_test = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/synthetic_data/data_20251219__level_1__subclasses_None/test_data.csv")["label"].to_list()

X_train = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/synthetic_data/data_20251222__level_1__subclasses_None__level_descriptions_3/train_data.csv")["text"].to_list()
y_train = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/synthetic_data/data_20251222__level_1__subclasses_None__level_descriptions_3/train_data.csv")["label"].to_list()
X_test = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/synthetic_data/data_20251222__level_1__subclasses_None__level_descriptions_3/test_data.csv")["text"].to_list()
y_test = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/synthetic_data/data_20251222__level_1__subclasses_None__level_descriptions_3/test_data.csv")["label"].to_list()

# prompt 1
data_path = "projects/nace_classification/nace_report_topic_analysis/data/synthetic_data/data_20251219__level_1__subclasses_None"
# prompt 2
data_path = "projects/nace_classification/nace_report_topic_analysis/data/synthetic_data/data_20251222__level_1__subclasses_None__level_descriptions_3"
# prompt 3
data_path = "projects/nace_classification/nace_report_topic_analysis/data/synthetic_data/data_20251218__level_1__subclasses_None__prompts_3__few_shot"
# prompt 4
data_path = "projects/nace_classification/nace_report_topic_analysis/data/synthetic_data/data_20251218__level_1__subclasses_None__prompts_4__few_shot"
# prompt 5
data_path = "projects/nace_classification/nace_report_topic_analysis/data/synthetic_data/data_20251218__level_1__subclasses_None__prompts_5__few_shot"
# prompt 5 with topics
data_path = "projects/nace_classification/nace_report_topic_analysis/data/synthetic_data/data_20251218__level_1__subclasses_None__prompts_5__few_shot__from_lvl_4_topics"

df_train = pd.read_csv(os.path.join(data_path, "train_data.csv"))
df_test = pd.read_csv(os.path.join(data_path, "test_data.csv"))

X_train = df_train["text"].to_list()
y_train = df_train["label"].to_list()
X_test = df_test["text"].to_list()
y_test = df_test["label"].to_list()

In [7]:
# make a tf idf classifier for secondary NACE codes

def make_fast_text_clf():
    return make_pipeline(
        TfidfVectorizer(ngram_range=(1,1), min_df=3, max_features=7000),
        SGDClassifier(loss="log_loss", alpha=1e-5, max_iter=1400, tol=1e-3, random_state=0)
    )

sec_clf = make_fast_text_clf().fit(X_train, y_train)

In [8]:
# test on test set
y_pred = sec_clf.predict(X_test)
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           A       0.45      0.43      0.44       100
           C       0.45      0.47      0.46       100

    accuracy                           0.45       200
   macro avg       0.45      0.45      0.45       200
weighted avg       0.45      0.45      0.45       200



In [9]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(ngram_range=(1,2), stop_words="english")
X = cv.fit_transform(X_train)

In [10]:
import numpy as np
from collections import defaultdict

def top_ngrams_per_class(
    clf_pipeline,
    X,
    y,
    top_k=20
):
    """
    Extract top TF-IDF n-grams per class via class centroids.
    
    Args:
        clf_pipeline: fitted sklearn Pipeline (TfidfVectorizer + classifier)
        X: list of texts
        y: list / array of labels
        top_k: number of top n-grams per class
    """
    vectorizer = clf_pipeline.named_steps["tfidfvectorizer"]
    
    # Transform texts
    X_tfidf = vectorizer.transform(X)
    feature_names = np.array(vectorizer.get_feature_names_out())

    # Group indices by class
    class_indices = defaultdict(list)
    for i, label in enumerate(y):
        class_indices[label].append(i)

    # Compute centroids + top features
    results = {}
    for label, idxs in class_indices.items():
        centroid = X_tfidf[idxs].mean(axis=0)
        centroid = np.asarray(centroid).ravel()

        top_idx = np.argsort(centroid)[-top_k:][::-1]
        results[label] = list(zip(feature_names[top_idx], centroid[top_idx]))

    return results

In [11]:
top_terms = top_ngrams_per_class(
    sec_clf,
    X_train,
    y_train,
    top_k=15
)

for cls, terms in top_terms.items():
    print(f"\nClass {cls}")
    for term, score in terms:
        print(f"  {term:<25} {score:.4f}")


Class C
  our                       0.1083
  to                        0.1026
  and                       0.0910
  the                       0.0862
  of                        0.0743
  we                        0.0677
  that                      0.0547
  with                      0.0449
  in                        0.0438
  local                     0.0420
  sustainable               0.0420
  for                       0.0391
  practices                 0.0391
  by                        0.0373
  while                     0.0366

Class A
  our                       0.1002
  to                        0.0959
  the                       0.0942
  and                       0.0910
  of                        0.0761
  we                        0.0628
  that                      0.0575
  in                        0.0480
  with                      0.0443
  local                     0.0435
  by                        0.0405
  for                       0.0389
  sustainable               0.0383
  

In [12]:
df_train[df_train["label"] == "A"]["text"].apply(lambda x: "agriculture" in x.lower()).mean()

0.12666666666666668

In [13]:
df_train[df_train["label"] == "C"]["text"].apply(lambda x: "manufacturing" in x.lower()).mean()

0.006666666666666667